# Notifiable Queue Stage

A `NotifiableQueueStage` makes agents line up at a sequence of positions and only advance when the simulation explicitly releases them.
Create it with `simulation.add_queue_stage(positions)`, retrieve the live handle with `simulation.get_stage(id)`, check occupancy with `queue.count_enqueued()`, and release agents with `queue.pop(n)`.
This notebook shows a queue that feeds directly into an exit, releasing two agents every 100 iterations.

See {doc}`Object model & lifecycle </concepts/object_model>` for how stages fit into the simulation lifecycle.

In [ ]:
import pathlib

import jupedsim as jps
import shapely

# 20 m x 10 m room.
geometry = shapely.Polygon([(0, 0), (20, 0), (20, 10), (0, 10)])

trajectory_file = pathlib.Path("queues.sqlite")
simulation = jps.Simulation(
    model=jps.CollisionFreeSpeedModel(),
    geometry=geometry,
    trajectory_writer=jps.SqliteTrajectoryWriter(
        output_file=trajectory_file, commit_every_nth_write=1
    ),
)

# Queue: agents line up at these positions and wait to be released.
queue_id = simulation.add_queue_stage([(7, 5), (7.4, 5), (7.8, 5)])
queue = simulation.get_stage(queue_id)

exit_id = simulation.add_exit_stage([(19, 4), (20, 4), (20, 6), (19, 6)])

# Wire queue → exit with a fixed transition.
journey = jps.JourneyDescription([queue_id, exit_id])
journey.set_transition_for_stage(
    queue_id, jps.Transition.create_fixed_transition(exit_id)
)
journey_id = simulation.add_journey(journey)

# Place 12 agents on the left side.
n_agents = 12
positions = jps.distributions.distribute_by_number(
    polygon=shapely.Polygon([(0.5, 0.5), (4, 0.5), (4, 9.5), (0.5, 9.5)]),
    number_of_agents=n_agents,
    distance_to_agents=0.4,
    distance_to_polygon=0.2,
    seed=1,
)
for position in positions:
    simulation.add_agent(
        jps.CollisionFreeSpeedModelAgentParameters(
            journey_id=journey_id,
            stage_id=queue_id,
            position=position,
            radius=0.12,
        )
    )

# Release 2 agents from the queue every 100 iterations.
while simulation.agent_count() > 0 and simulation.iteration_count() < 10_000:
    if simulation.iteration_count() % 100 == 0 and queue.count_enqueued() > 0:
        queue.pop(2)
    simulation.iterate()

print(
    f"Done in {simulation.iteration_count()} iterations. Trajectories: {trajectory_file}"
)

## Watch it

In [ ]:
from jupedsim.internal.notebook_utils import animate, read_sqlite_file

trajectory_data, walkable_area = read_sqlite_file(trajectory_file)
animate(trajectory_data, walkable_area)

## Try one change

Change the batch size in `queue.pop(2)` to `queue.pop(1)` (one at a time) or the release interval from `% 100` to `% 50` (twice as often) and re-run. Watch how throughput changes in the animation.

## See also

- {doc}`Waiting Set Stage </notebooks/fundamentals/06_waiting>` — hold agents in place and release them all at once.
- {doc}`Journeys and Transitions </notebooks/fundamentals/07_journeys_transitions>` — chain stages with decisions.
- {doc}`Object model & lifecycle </concepts/object_model>`.
- [jupedsim-scenarios](https://scenarios.jupedsim.org) — parameter sweeps and Monte-Carlo runs.